In [ ]:
# import sys

# if 'google.colab' in sys.modules:
#     print("Running in Google Colab")
#     print("Version:", sys.version)

# else:
#     print("Not running in Google Colab")
#     print("Version:", sys.version)

In [1]:
!pip install gcsfs pyarrow   # for GCS parquet upload. running in terminal

In [ ]:
import pandas as pd

path = "gs://emotor-dataset-raw/current_temp_short/0Nm_BPFI_03.parquet"
df = pd.read_parquet(path)

"""
    Channel mapping for eMotor dataset (current + temperature)

    Log/cDAQ9185-1F486B5Mod1/ai0 -> temp_A
    Log/cDAQ9185-1F486B5Mod1/ai1 -> temp_B
    Log/cDAQ9185-1F486B5Mod2/ai0 -> current_u
    Log/cDAQ9185-1F486B5Mod2/ai2 -> current_v
    Log/cDAQ9185-1F486B5Mod2/ai3 -> current_w
"""

print(df.shape)
print(df.columns)
df.head()

In [ ]:
path = "gs://emotor-dataset-raw/vibration_temp_short/0Nm_BPFI_03.parquet"
df_vibration = pd.read_parquet(path)

"""
    Channel mapping for eMotor dataset (vibration)

    ch1 -> vib_x_A,
    ch2 -> vib_y_A,
    ch3 -> vib_x_B,
    ch4 -> vib_y_B
"""

print(df_vibration.shape)
print(df_vibration.columns)
df_vibration.head()

### Parse torque and label from filename

In [ ]:
import re

def parse_metadata(filename: str):
    """
    Ejemplo filename: 0Nm_BPFI_03.parquet
    """
    torque = int(re.search(r"(\d)Nm", filename).group(1))

    if "Normal" in filename:
        label = "Normal"
    elif "BPFI" in filename:
        label = "BPFI"
    elif "BPFO" in filename:
        label = "BPFO"
    else:
        label = None

    return torque, label

### Load and std dataframe: current + temperature

In [ ]:
import pandas as pd

def load_current_temp(path):
    df = pd.read_parquet(path)

    df = df.rename(columns={
        "Log/cDAQ9185-1F486B5Mod2/ai0": "current_u",
        "Log/cDAQ9185-1F486B5Mod2/ai2": "current_v",
        "Log/cDAQ9185-1F486B5Mod2/ai3": "current_w",
        "Log/cDAQ9185-1F486B5Mod1/ai0": "temp_A",
        "Log/cDAQ9185-1F486B5Mod1/ai1": "temp_B"
    })

    return df.reset_index(drop=True)

### Load and std dataframe: vibration

In [ ]:
def load_vibration(path):
    df = pd.read_parquet(path)

    df = df.rename(columns={
        "ch1": "vib_x_A",
        "ch2": "vib_y_A",
        "ch3": "vib_x_B",
        "ch4": "vib_y_B"
    })

    return df[["vib_x_A", "vib_y_A", "vib_x_B", "vib_y_B"]].reset_index(drop=True)

### Merge current/temp and vibration dataframes

In [ ]:
def merge_signals(df_curr, df_vib):
    n = min(len(df_curr), len(df_vib))

    df = pd.concat(
        [df_curr.iloc[:n], df_vib.iloc[:n]],
        axis=1
    )

    return df


### Windowing

In [ ]:
import numpy as np

def sliding_window(df, window_size, step_size):
    """
    df: DataFrame con señales
    window_size: nº de samples por ventana
    step_size: salto entre ventanas
    """
    windows = []

    for start in range(0, len(df) - window_size + 1, step_size):
        end = start + window_size
        window = df.iloc[start:end]
        windows.append(window)

    return windows

### Feature extraction (ML)
##### **mean:** promedio aritmético de todos los valores de la señal en un tiempo determinado. En una señal de corriente alterna (AC)     perfecta, la media es cero; Si no es cero, indica una componente de corriente continua (DC)

##### **std:** mide la variación los datos respecto a la media. En señales eléctricas sin componente DC, el valor de la desviación estándar es prácticamente igual al valor RMS.

##### **RMS:** valor eficaz de la corriente, es decir, el valor de una corriente continua que produciría la misma disipación de calor que la señal alterna

##### **kurtosis:** mide qué tan "puntiaguda" o "achatada" es la distribución de la señal. En análisis de vibraciones o corrientes, una curtosis alta suele indicar la presencia de picos transitorios o impactos (como un rodamiento dañado o un arco eléctrico), ya que hay valores extremos fuera de lo común.

##### **skew:** asimetría. Indica si la señal es simétrica respecto a la media. Una señal balanceada tiene asimetría cero. Si hay fallos en un rectificador, por ejemplo, la señal puede "cargarse" más hacia un lado, aumentando la asimetría.

In [ ]:
# Define fast and slow signal names
FAST_SIGNALS = [
    "current_u", "current_v", "current_w",
    "vib_x_A", "vib_y_A", "vib_x_B", "vib_y_B"
]

SLOW_SIGNALS = [
    "temp_A", "temp_B"
]

In [ ]:
# Feature extraction functions
from scipy.stats import kurtosis, skew

def extract_time_features(window):
    features = {}

    # Señales rápidas
    for col in FAST_SIGNALS:
        x = window[col].values
        features[f"{col}_mean"] = np.mean(x)
        features[f"{col}_std"] = np.std(x)
        features[f"{col}_rms"] = np.sqrt(np.mean(x**2))
        features[f"{col}_kurtosis"] = kurtosis(x)
        features[f"{col}_skew"] = skew(x)

    # Señales lentas (temperatura)
    for col in SLOW_SIGNALS:
        x = window[col].values
        features[f"{col}_mean"] = np.mean(x)
        features[f"{col}_std"] = np.std(x)

    return features

In [ ]:
# from scipy.stats import kurtosis, skew

# def extract_time_features(window):
#     features = {}

#     for col in window.columns:
#         x = window[col].values

#         features[f"{col}_mean"] = np.mean(x)
#         features[f"{col}_std"] = np.std(x)
#         features[f"{col}_rms"] = np.sqrt(np.mean(x**2))
#         features[f"{col}_kurtosis"] = kurtosis(x)
#         features[f"{col}_skew"] = skew(x)

#     return features


### All pipeline

In [ ]:
def process_file(
    curr_path,
    vib_path,
    filename,
    window_size,
    step_size
):
    df_curr = load_current_temp(curr_path)
    df_vib  = load_vibration(vib_path)

    df = merge_signals(df_curr, df_vib)

    torque, label = parse_metadata(filename)
    if label is None:
        return None

    windows = sliding_window(df, window_size, step_size)

    rows = []
    for w in windows:
        feats = extract_time_features(w)
        feats["label"] = label
        feats["torque_nm"] = torque
        feats["source_file"] = filename
        rows.append(feats)

    return pd.DataFrame(rows)


In [ ]:
SAMPLING_RATE = 10000   # ejemplo (ajustable)
WINDOW_SECONDS = 1.0

WINDOW_SIZE = int(SAMPLING_RATE * WINDOW_SECONDS)
STEP_SIZE   = int(WINDOW_SIZE * 0.5)

In [ ]:
df_ml = process_file("gs://emotor-dataset-raw/current_temp_short/0Nm_BPFI_03.parquet",
                     "gs://emotor-dataset-raw/vibration_temp_short/0Nm_BPFI_03.parquet",
                     "0Nm_BPFI_03",
                     2_000,
                     1_000)

print(df_ml.shape)
print(df_ml.columns)
df_ml.head()

### Feature engineering in freq: FFT features

In [ ]:
def compute_fft(x, fs):
    """FFT normalizada (solo magnitud positiva)."""
    x = x - np.mean(x)
    X = np.fft.rfft(x)
    freqs = np.fft.rfftfreq(len(x), d=1/fs)
    mag = np.abs(X)
    return freqs, mag

In [ ]:
# Features espectrales por señal

def spectral_features(x, fs):
    freqs, mag = compute_fft(x, fs)

    power = mag**2
    total_energy = np.sum(power)

    # Bandas (fracciones de Nyquist)
    nyq = fs / 2
    low_band  = freqs <= nyq * 0.2
    mid_band  = (freqs > nyq * 0.2) & (freqs <= nyq * 0.5)
    high_band = freqs > nyq * 0.5

    energy_low  = np.sum(power[low_band])
    energy_mid  = np.sum(power[mid_band])
    energy_high = np.sum(power[high_band])

    # Entropía espectral
    psd_norm = power / (total_energy + 1e-12)
    spec_entropy = -np.sum(psd_norm * np.log(psd_norm + 1e-12))

    return {
        "spec_energy_total": total_energy,
        "spec_energy_low": energy_low,
        "spec_energy_mid": energy_mid,
        "spec_energy_high": energy_high,
        "spec_entropy": spec_entropy
    }


In [ ]:
# Integrar FFT al extractor por ventana

DYNAMIC_COLS = [
    "current_u", "current_v", "current_w",
    "vib_x_A", "vib_y_A", "vib_x_B", "vib_y_B"
]

def extract_freq_features(window, fs):
    feats = {}
    for col in DYNAMIC_COLS:
        x = window[col].values
        spec = spectral_features(x, fs)
        for k, v in spec.items():
            feats[f"{col}_{k}"] = v
    return feats


In [ ]:
# Unir tiempo + frecuencia en el extractor de características

def extract_features(window, fs):
    feats_time = extract_time_features(window)
    feats_freq = extract_freq_features(window, fs)
    feats_time.update(feats_freq)
    return feats_time

In [ ]:
# Update pipeline para usar extractor combinado

def process_file_fft(
    curr_path,
    vib_path,
    filename,
    window_size,
    step_size,
    fs
):
    df_curr = load_current_temp(curr_path)
    df_vib  = load_vibration(vib_path)
    df = merge_signals(df_curr, df_vib)

    torque, label = parse_metadata(filename)
    if label is None:
        return None

    windows = sliding_window(df, window_size, step_size)

    rows = []
    for w in windows:
        feats = extract_features(w, fs)
        feats["label"] = label
        feats["torque_nm"] = torque
        feats["source_file"] = filename
        rows.append(feats)

    return pd.DataFrame(rows)


In [ ]:
df_ml_global = process_file_fft("gs://emotor-dataset-raw/current_temp_short/0Nm_BPFI_03.parquet",
                     "gs://emotor-dataset-raw/vibration_temp_short/0Nm_BPFI_03.parquet",
                     "0Nm_BPFI_03",
                     2_000,
                     1_000,
                     25600)

print(df_ml_global.shape)
print(df_ml_global.columns)
df_ml_global.head()

In [ ]:
df_ml_global.describe()

In [ ]:
df_ml_global.info()

### 4. Batch processing

In [ ]:
# Define base paths
from pathlib import Path
import gcsfs

fs = gcsfs.GCSFileSystem()

CURR_PATH = "gs://emotor-dataset-raw/current_temp_short/"
VIBR_PATH = "gs://emotor-dataset-raw/vibration_temp_short/"

curr_files = [
    Path(f).name for f in fs.ls(CURR_PATH)
    if f.endswith(".parquet")
]

vibr_files = [
    Path(f).name for f in fs.ls(VIBR_PATH)
    if f.endswith(".parquet")
]

common_files = sorted(set(curr_files).intersection(vibr_files))
print(f"Archivos comunes: {len(common_files)}")

In [ ]:
# List files in GCS buckets
def is_valid_file(filename: str) -> bool:
    return any(k in filename for k in ["Normal", "BPFI", "BPFO"])


selected_files = [
    f for f in common_files
    if is_valid_file(f)
]

print(f"Archivos seleccionados: {len(selected_files)}")

In [ ]:
from concurrent.futures import ProcessPoolExecutor, as_completed

def process_single_file(filename):
    curr_path = CURR_PATH + filename
    vib_path  = VIBR_PATH + filename

    try:
        df_part = process_file_fft(
            curr_path=curr_path,
            vib_path=vib_path,
            filename=filename,
            window_size=2_000,
            step_size=1_000,
            fs=25_600
        )
        return df_part

    except Exception as e:
        return f"ERROR::{filename}::{e}"

In [ ]:
import os

dfs = []
errors = []

MAX_WORKERS = max(1, os.cpu_count() - 1)

with ProcessPoolExecutor(max_workers=MAX_WORKERS) as executor:
    futures = {
        executor.submit(process_single_file, fname): fname
        for fname in selected_files
    }

    for future in as_completed(futures):
        result = future.result()

        if isinstance(result, pd.DataFrame):
            dfs.append(result)
        else:
            errors.append(result)

In [ ]:
dfs = []

for filename in selected_files:
    curr_path = CURR_PATH + filename
    vib_path  = VIBR_PATH + filename

    try:
        df_part = process_file_fft(
            curr_path=curr_path,
            vib_path=vib_path,
            filename=filename,
            window_size=2_000,
            step_size=1_000,
            fs=25_600
        )

        if df_part is not None:
            dfs.append(df_part)

    except Exception as e:
        print(f"⚠️ Error en {filename}: {e}")


In [ ]:
# Filter files for bearing fault types
def is_valid_file(fname):
    return (
        ("Normal" in fname) or
        ("BPFI" in fname) or
        ("BPFO" in fname)
    )

valid_files = [f for f in files if is_valid_file(f)]
len(valid_files)

In [ ]:
print("Total curr:", len(curr_files))
print("Total vib :", len(vib_files))
print("Intersect :", len(files))
print("Valid     :", len(valid_files))

In [ ]:
# Batch processing: Process all files and aggregate results
from tqdm import tqdm

dfs = []

for fname in tqdm(valid_files):
    curr_path = BASE_CURR + fname
    vib_path  = BASE_VIB  + fname

    try:
        df_file = process_file_fft(
            curr_path,
            vib_path,
            fname.replace(".parquet", ""),
            WINDOW_SIZE,
            STEP_SIZE,
            FS
        )
        if df_file is not None:
            dfs.append(df_file)
    except Exception as e:
        print(f"❌ Error en {fname}: {e}")

In [ ]:
# Aggregate all data
df_ml_global = pd.concat(dfs, ignore_index=True)

df_ml_global.shape